# 02 — Feature Ablation

Measures how much each of the four feature families (provenance, temporal, profile consistency, causal attribution) contributes to DriftGuard's three-class F1, by re-training the classifier with each family zeroed out.

Requires `data/benchmark.jsonl` (run `scripts/generate_benchmark.py` first).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from driftguard.benchmark.labelling import load_benchmark
from driftguard.detector.features import FeatureExtractor, FEATURE_DIM
from driftguard.detector.classifier import DriftGuardClassifier
from eval.metrics import compute_all_metrics

BENCHMARK_PATH = "../data/benchmark.jsonl"
entries = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(entries)} entries")

In [ ]:
extractor = FeatureExtractor()
records = [
    {
        "content": e.content,
        "provenance": e.provenance,
        "write_history": e.write_history,
        "session_id": e.session_id,
        "last_query": e.last_query,
        "memory_entries": e.memory_entries,
    }
    for e in entries
]
X = extractor.extract_batch(records)
y = np.array([e.label_int for e in entries])
X.shape, y.shape

## Feature family index ranges

See `driftguard/detector/features.py::FeatureExtractor` docstring for the full layout.

In [ ]:
FEATURE_FAMILIES = {
    "provenance":   slice(0, 3),
    "temporal":     slice(3, 7),
    "consistency":  slice(7, 12),
    "causal":       slice(12, 18),
}
assert sum((s.stop - s.start) for s in FEATURE_FAMILIES.values()) == FEATURE_DIM

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]


def run_ablation(zero_family: str | None) -> dict:
    """Train/evaluate with one feature family zeroed out (None = full feature set)."""
    Xtr, Xte = X_train.copy(), X_test.copy()
    if zero_family is not None:
        s = FEATURE_FAMILIES[zero_family]
        Xtr[:, s] = 0.0
        Xte[:, s] = 0.0

    clf = DriftGuardClassifier(backend="random_forest", n_estimators=200)
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xte)
    y_proba = clf.predict_proba(Xte)
    metrics = compute_all_metrics(y_test, y_pred, y_proba)
    return {
        "ablated_family": zero_family or "none (full model)",
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "FAR": metrics["FAR"],
        "MAR": metrics["MAR"],
    }

In [ ]:
results = [run_ablation(None)] + [run_ablation(fam) for fam in FEATURE_FAMILIES]
ablation_df = pd.DataFrame(results)
ablation_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ablation_df["ablated_family"], ablation_df["f1_macro"], color="#457b9d")
ax.axhline(ablation_df.loc[0, "f1_macro"], color="gray", linestyle="--", label="full model")
ax.set_ylabel("F1 (macro)")
ax.set_title("Feature-family ablation — impact on macro F1")
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Per-feature importance (full model)

In [ ]:
feature_names = [
    "origin_index", "trust_level", "session_age_hours",
    "writes_last_10", "writes_last_30", "inter_write_interval_mean_min", "burst_score",
    "cosine_sim_to_centroid", "novelty_score", "same_session_writes", "unique_origins_count", "repeated_hash_flag",
    "query_content_overlap", "write_length_chars", "contains_instruction", "contains_url", "avg_word_length", "exclamation_ratio",
]

full_clf = DriftGuardClassifier(backend="random_forest", n_estimators=200)
full_clf.fit(X_train, y_train, feature_names=feature_names)
importances = full_clf.feature_importances()
importances

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importances["feature"], importances["importance"], color="#2a9d8f")
ax.invert_yaxis()
ax.set_xlabel("Importance")
ax.set_title("DriftGuard feature importances (RandomForest)")
plt.tight_layout()
plt.show()